In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import tensorflow as tf
from sklearn.preprocessing import MinMaxScaler
import os

def load_test_data(json_file_path):
    """
    Test verilerini JSON dosyasından yükler.
    
    Args:
        json_file_path (str): Test JSON dosyasının yolu
        
    Returns:
        pandas.DataFrame: Test verisi
    """
    try:
        with open(json_file_path, 'r') as f:
            data = json.load(f)
            
        # JSON verisini pandas DataFrame'e dönüştür
        df = pd.DataFrame(data['packets'])
        print(f"Test verisi başarıyla yüklendi. {len(df)} paket bulundu.")
        return df
    except Exception as e:
        print(f"Test verisi yüklenirken hata oluştu: {str(e)}")
        return None

def preprocess_data(df):
    """
    Test verisini modele uygun şekilde ön işler.
    
    Args:
        df (pandas.DataFrame): Ham test verisi
        
    Returns:
        pandas.DataFrame: İşlenmiş test verisi
    """
    try:
        # İlgili sütunları seç
        required_columns = ["hour", "minute", "second", "size", "source_port", "destination_port", "ttl", "protocol", "tcp_flags"]
        available_columns = [col for col in required_columns if col in df.columns]
        
        if not set(["hour", "minute", "second", "size"]).issubset(set(available_columns)):
            print("Uyarı: Bazı gerekli sütunlar eksik.")
        
        # Mevcut sütunları kullan
        df_processed = df[available_columns].copy()
        
        # Protokolleri one-hot encode et
        if "protocol" in df_processed.columns:
            df_processed = pd.get_dummies(df_processed, columns=["protocol"], prefix="protocol")
        
        # TCP bayraklarını işle
        if "tcp_flags" in df_processed.columns:
            df_processed["tcp_flags"] = df_processed["tcp_flags"].astype(str)
            df_processed["SYN"] = df_processed["tcp_flags"].apply(lambda x: int("SYN=true" in x))
            df_processed["ACK"] = df_processed["tcp_flags"].apply(lambda x: int("ACK=true" in x))
            df_processed["FIN"] = df_processed["tcp_flags"].apply(lambda x: int("FIN=true" in x))
            df_processed.drop(columns=["tcp_flags"], inplace=True)
        
        # TTL sütununu doldur
        if "ttl" in df_processed.columns and df_processed["ttl"].isna().any():
            df_processed["ttl"].fillna(df_processed["ttl"].median(), inplace=True)
        
        # NaN değerlerini doldur
        for col in df_processed.columns:
            if df_processed[col].dtype != 'object' and df_processed[col].isna().any():
                print(f"Uyarı: {col} sütununda NaN değerler var. Medyan ile dolduruluyor.")
                df_processed[col].fillna(df_processed[col].median(), inplace=True)
                
        return df_processed
    except Exception as e:
        print(f"Veri ön işleme sırasında hata oluştu: {str(e)}")
        return None

def detect_anomalies(model_path, test_data_path, sequence_length=50, threshold=None):
    """
    Test verisindeki anomalileri tespit eder.
    
    Args:
        model_path (str): Eğitilmiş model dosyasının yolu
        test_data_path (str): Test veri dosyasının yolu
        sequence_length (int): Dizilerin uzunluğu
        threshold (float): Anomali eşik değeri (None ise otomatik hesaplanır)
        
    Returns:
        list: Anomali indeksleri ve skoru
    """
    try:
        # 1. Modeli yükle
        print(f"Model yükleniyor: {model_path}")
        model = tf.keras.models.load_model(model_path)
        print("Model başarıyla yüklendi.")
        
        # 2. Test verisini yükle ve ön işle
        raw_df = load_test_data(test_data_path)
        if raw_df is None:
            return None
            
        df = preprocess_data(raw_df)
        if df is None:
            return None
            
        # 3. Veriyi normalize et
        scaler = MinMaxScaler()
        df_scaled = scaler.fit_transform(df)
        
        # 4. Rekonstrüksiyon hatalarını hesapla
        # 4.1 Tek tek paketler için anomali skorunu hesapla
        individual_scores = []
        for i in range(len(df_scaled)):
            # İndeks dışına çıkma durumunu kontrol et
            if i + sequence_length > len(df_scaled):
                break
                
            # Diziyi oluştur
            sequence = df_scaled[i:i+sequence_length]
            sequence = np.expand_dims(sequence, axis=0)  # Batch boyutunu ekle
            
            # Yeniden oluştur
            reconstructed = model.predict(sequence, verbose=0)
            
            # MSE hesapla
            mse = np.mean(np.power(sequence - reconstructed, 2))
            individual_scores.append((i, mse))
        
        # 5. Eşik değeri belirle
        if threshold is None:
            mse_values = [score for _, score in individual_scores]
            threshold = np.percentile(mse_values, 95)  # 95. yüzdelik
            print(f"Otomatik belirlenen anomali eşik değeri: {threshold:.6f}")
        else:
            print(f"Belirtilen anomali eşik değeri: {threshold:.6f}")
            
        # 6. Anomalileri belirle
        anomalies = []
        for idx, score in individual_scores:
            if score > threshold:
                anomalies.append((idx, score))
                
        # 7. Anomalileri raporla
        print(f"\nToplam {len(anomalies)} anomali tespit edildi.")
        print(f"Test verisinde {len(raw_df)} paket, {len(individual_scores)} sıralı dizi var.")
        
        if anomalies:
            print("\n--- Tespit Edilen Anomaliler ---")
            for idx, score in anomalies:
                print(f"Paket indeksi: {idx}, Anomali skoru: {score:.6f}")
                
                # İlgili paketin özelliklerini göster
                packet = raw_df.iloc[idx]
                print(f"Zaman: {packet['hour']:02d}:{packet['minute']:02d}:{packet['second']:02d}")
                print(f"Paket boyutu: {packet['size']} bayt")
                if 'protocol' in packet:
                    print(f"Protokol: {packet['protocol']}")
                if 'source_ip' in packet and 'destination_ip' in packet:
                    print(f"Kaynak IP: {packet['source_ip']}, Hedef IP: {packet['destination_ip']}")
                if 'source_port' in packet and 'destination_port' in packet:
                    print(f"Kaynak port: {packet['source_port']}, Hedef port: {packet['destination_port']}")
                if 'tcp_flags' in packet:
                    print(f"TCP bayrakları: {packet['tcp_flags']}")
                print("------------------------")
            
            # 8. Görselleştirme
            plt.figure(figsize=(14, 6))
            
            # Anomali skorları
            scores = [score for _, score in individual_scores]
            plt.plot(scores, label='Anomali Skoru')
            
            # Eşik değeri
            plt.axhline(y=threshold, color='r', linestyle='--', label=f'Eşik Değeri: {threshold:.6f}')
            
            # Anomalileri işaretle
            anomaly_indices = [idx for idx, _ in anomalies]
            anomaly_scores = [score for _, score in anomalies]
            plt.scatter(anomaly_indices, anomaly_scores, color='red', marker='o', label='Anomaliler')
            
            plt.title('Paket Anomali Skorları')
            plt.xlabel('Paket İndeksi')
            plt.ylabel('Anomali Skoru (MSE)')
            plt.legend()
            plt.grid(True, alpha=0.3)
            plt.tight_layout()
            plt.savefig("anomali_skorlari.png")
            print("Anomali skorları grafiği 'anomali_skorlari.png' dosyasına kaydedildi.")
            
        return anomalies
            
    except Exception as e:
        print(f"Anomali tespiti sırasında bir hata oluştu: {str(e)}")
        return None

def main():
    """
    Ana fonksiyon
    """
    print("Anomali Tespit Aracı")
    print("====================")
    
    # Model dosyasını seç
    model_path = "C:/Users/edadd/Desktop/bitirme/model/just_lstm_autoencoder.h5"
    if not os.path.exists(model_path):
        print(f"Hata: {model_path} dosyası bulunamadı.")
        return
    
    # Test verisini seç
    test_data_path = "C:/Users/edadd/Desktop/bitbitbit/test_verisi.json"
    if not os.path.exists(test_data_path):
        print(f"Hata: {test_data_path} dosyası bulunamadı.")
        return
    
    # Anomali tespitini gerçekleştir
    detect_anomalies(model_path, test_data_path)
    
    print("\nAnomaliler başarıyla tespit edildi!")

if __name__ == "__main__":
    main()

Anomali Tespit Aracı
Model yükleniyor: C:/Users/edadd/Desktop/bitirme/model/just_lstm_autoencoder.h5
Anomali tespiti sırasında bir hata oluştu: Could not locate function 'mse'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'keras.metrics', 'class_name': 'function', 'config': 'mse', 'registered_name': 'mse'}

Anomaliler başarıyla tespit edildi!
